# gptlab runner

This notebook only starts the job runner. All logic lives in the GitHub repo, so this notebook never needs to change.

Each time you click **Save Version -> Save & Run All (Commit)** it:
1. clones the repo (`main` branch) and installs the requirements,
2. runs the next job(s) from `runs/queue.yaml`,
3. pushes logs to the `results` branch and checkpoints to the Hugging Face Hub,
4. stops cleanly about 11 hours after it started.

Setup (secrets, GPU, internet): see `RUNNING.md` in the repo.

In [ ]:
import os, shutil, subprocess, sys, time

SESSION_START = time.time()  # the runner stops cleanly 11 hours after this moment
REPO_URL = "https://github.com/MelvTheGoat/LLM.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/LLM"
WORK_DIR = "/tmp/gptlab-work"  # scratch space for data and checkpoints (not saved as output)

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
env = dict(os.environ, SESSION_START=str(SESSION_START))
for name in ["GH_TOKEN", "HF_TOKEN"]:
    try:
        env[name] = secrets.get_secret(name)
    except Exception:
        raise SystemExit(f"Secret {name} is missing: add it under Add-ons > Secrets and tick it for this notebook.")


def run(cmd, cwd=None):
    """Run a command and show its output live. Never prints the environment (it holds the tokens)."""
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    return p.wait()


subprocess.run(["rm", "-rf", REPO_DIR])
assert run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR]) == 0, "git clone failed"
assert run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO_DIR) == 0, "pip failed"
if shutil.which("nvidia-smi"):  # CPU sessions have no nvidia-smi
    run(["nvidia-smi"])
code = run([sys.executable, "-m", "gptlab.runner", "--work", WORK_DIR], cwd=REPO_DIR)
print("runner exit code:", code)
if code != 0:
    raise SystemExit(code)  # marks this version as failed, so problems are easy to spot
